# XJTU action-conditioned digital twin validation

Validate future SOH, delivered energy and EOL for held-out batteries under their recorded 2C or 3C charging protocol. Both groups use 1C discharge and a nominal 2 Ah capacity. Each simulated cycle updates SOH from the preceding state and recorded charge rate.

Open the notebook inside the project and select its Python kernel. Run all cells in order. It reads the extracted Dataset/XJTU folder or XJTU.zip and the saved Notebook 01 and 07 reports. Notebook 11 is an optional reference. The audit processes one raw battery at a time and uses one CPU thread. The output ZIP is saved under reports.

In [1]:
import os

for name in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "BLIS_NUM_THREADS"):
    os.environ[name] = "1"

import json
import pickle
from collections import defaultdict
from contextlib import contextmanager, nullcontext
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from threadpoolctl import threadpool_limits

DATASET_PATH = None  # Defaults to the extracted Dataset/XJTU folder, then Dataset/XJTU.zip.
OUTPUT_ROOT = None  # Defaults to reports under the project.
PILOT_CELLS = ("XJTU_2C_battery-7", "XJTU_3C_battery-11")
ANCHORS = (50, 100)
HORIZONS = (1, 10, 50)
RATES = (2, 3)
ALPHAS = (1, 10, 100)
EOL_SOH = 80.0
NOMINAL_AH = 2.0
MAX_STEPS = 600
WINDOW = 10
MIN_CELLS_IN_BIN = 2
NB01 = "lifeguard_soh_results"
NB07 = "lifeguard_digital_twin_validation_results"
NB11 = "lifeguard_rolling_forecasts_results"
OUTPUT_NAME = "lifeguard_action_conditioned_twin_results"

here = Path.cwd()
project = next((directory for directory in (here, *here.parents)
                if (directory / "README_LifeGuard_RL.md").is_file()
                and (directory / "reports").is_dir()), None)
if project is None:
    raise FileNotFoundError("Open the notebook from within the LifeGuard-RL project.")
reports = project / "reports"
dataset = Path(DATASET_PATH).expanduser() if DATASET_PATH else project / "Dataset" / "XJTU"
if DATASET_PATH is None and not dataset.is_dir():
    dataset = dataset.with_suffix(".zip")
output = (Path(OUTPUT_ROOT).expanduser() if OUTPUT_ROOT else reports) / OUTPUT_NAME


@contextmanager
def saved_csv(folder, filename):
    directory = reports / folder
    path = directory / filename
    if path.is_file():
        yield pd.read_csv(path)
        return
    archive = directory.with_suffix(".zip")
    if not archive.is_file():
        raise FileNotFoundError(f"Expected {path} or {archive}.")
    with ZipFile(archive) as bundle:
        files = [name for name in bundle.namelist() if Path(name).name == filename]
        if len(files) != 1:
            raise ValueError(f"Expected one {filename} in {archive}.")
        with bundle.open(files[0]) as file:
            yield pd.read_csv(file)


def unique_keys(frame, keys, name):
    if frame[keys].isna().any().any() or frame.duplicated(keys).any():
        raise ValueError(f"Missing or duplicated keys in {name}.")

In [2]:
def load_references():
    with saved_csv(NB07, "eol_audit.csv") as audit:
        audit = audit.set_index("cell_id", verify_integrity=True)
    expected_rates = {f"XJTU_{rate}C_battery-{number}": rate
                      for rate, count in ((2, 8), (3, 15)) for number in range(1, count + 1)}
    if audit.charge_rate_c.to_dict() != expected_rates:
        raise ValueError("Notebook 07 EOL audit does not match the fixed 2C and 3C cohort.")
    if (not audit.event_status.isin(["observed", "right_censored"]).all()
            or not audit.observed_eol_cycle.notna().eq(audit.event_status.eq("observed")).all()):
        raise ValueError("Invalid verified EOL status.")
    audit["endpoint"] = audit.observed_eol_cycle.fillna(audit.last_observed_cycle).astype(int)

    with saved_csv(NB07, "cell_splits.csv") as splits:
        unique_keys(splits, ["fold", "cell_id"], "Notebook 07 splits")
        folds = sorted(splits.fold.unique())
        test = splits.loc[splits.role.eq("test")].set_index("cell_id", verify_integrity=True)
        expected = {(fold, cell, "test" if test.loc[cell, "fold"] == fold else "train", rate)
                    for fold in folds for cell, rate in expected_rates.items()}
        keys = ["fold", "cell_id", "role", "charge_rate_c"]
        if len(folds) != 5 or set(splits[keys].itertuples(index=False, name=None)) != expected:
            raise ValueError("Expected five complete, disjoint Notebook 07 test groups.")
    audit["fold"] = test.fold

    with saved_csv(NB01, "cycle_quality.csv") as quality:
        quality = quality.loc[quality.cell_id.isin(audit.index)].copy()
    unique_keys(quality, ["cell_id", "cycle"], "Notebook 01 cycle quality")
    valid = np.isfinite(quality.discharge_capacity_ah) & quality.discharge_capacity_ah.gt(0)
    valid &= ~quality.status.isin(["unaligned_or_empty_arrays", "nonfinite_trace",
                                  "nonincreasing_timestamps", "missing_full_discharge"])
    quality["observed_soh_pct"] = quality.soh_pct.where(valid)
    quality["endpoint"] = quality.cell_id.map(audit.endpoint)
    quality = quality.loc[quality.cycle.le(quality.endpoint)]
    soh = {cell: group.set_index("cycle").observed_soh_pct.sort_index()
           for cell, group in quality.groupby("cell_id")}
    for cell, trace in soh.items():
        observed = trace[trace.le(EOL_SOH)]
        crossing = int(observed.index[0]) if len(observed) else None
        verified = audit.loc[cell, "observed_eol_cycle"]
        if crossing != (int(verified) if pd.notna(verified) else None):
            raise ValueError(f"Notebook 01 and Notebook 07 disagree on the first 80% crossing for {cell}.")

    with saved_csv(NB07, "soh_trajectories.csv") as rows:
        reference = rows.loc[rows.initializer.eq("measured") & rows.model.eq("soh_dependent")].copy()
    unique_keys(reference, ["cell_id", "landmark_cycle", "cycle"], "Notebook 07 measured trajectories")
    for cell, group in reference.groupby("cell_id"):
        np.testing.assert_allclose(soh[cell].reindex(group.cycle).to_numpy(),
                                   group.observed_soh_pct.to_numpy(), rtol=0, atol=1e-9)
    with saved_csv(NB07, "transition_calibration.csv") as table:
        calibration = table.loc[table.used].copy()
    with saved_csv(NB07, "rollout_starts.csv") as rows:
        starts = rows.loc[rows.initializer.eq("measured") & rows.model.eq("soh_dependent")].copy()
    unique_keys(starts, ["cell_id", "landmark_cycle"], "Notebook 07 measured starts")
    return audit, soh, reference, calibration, starts


cells, measured_soh, nb07_curves, nb07_fade, nb07_starts = load_references()
print(f"Loaded {len(cells)} batteries and {cells.observed_eol_cycle.notna().sum()} verified EOLs.")

Loaded 23 batteries and 22 verified EOLs.


## Raw trace and protocol audit

Discharge Wh is the trapezoidal integral of voltage times absolute discharge current, using intervals fully within the discharge segment. A complete trace must match the 1C discharge current, end near 2.5 V, and agree with the raw capacity within the greater of 0.01 Ah and 1%. The audit records missing or unusable labels.

The median charging current 30–90 seconds after onset identifies 2C or 3C within 5% of the expected current. A sustained switch has two consecutive classified cycles on both sides of an action change. The two pilot cells are read and checked first.

In [3]:
TRACE_FIELDS = ("time_in_s", "voltage_in_V", "current_in_A", "discharge_capacity_in_Ah")
SWITCH_COLUMNS = ("cell_id", "cycle", "from_rate_c", "to_rate_c")


class XJTUUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError(f"Unexpected global in XJTU data: {module}.{name}")


def discharge_integrals(time, volts, current):
    active = current < -0.1
    intervals = active[:-1] & active[1:]
    if not intervals.any():
        return None
    delta = np.diff(time)[intervals]
    power = volts * -current
    amp_hours = np.dot(delta, (-current[:-1] - current[1:])[intervals]) / 7200
    watt_hours = np.dot(delta, (power[:-1] + power[1:])[intervals]) / 7200
    return float(amp_hours), float(watt_hours)


def actual_charge_rate(time, current, start):
    seconds = time - time[start]
    stable = current[(seconds >= 30) & (seconds <= 90) & (current > 1)]
    if len(stable) < 5:
        return None, None
    amperes = float(np.median(stable))
    rate = next((rate for rate in RATES if abs(amperes - NOMINAL_AH * rate)
                 <= 0.05 * NOMINAL_AH * rate), None)
    return rate, amperes


def audit_cycle(cycle):
    result = {
        "observed_charge_rate_c": None, "cc_current_a": None, "discharge_current_a": None,
        "charge_to_cutoff_s": None, "recorded_discharge_ah": None, "integrated_discharge_ah": None,
        "delivered_wh": None, "energy_status": "invalid_trace",
    }
    traces = [np.asarray(cycle[field], dtype=float) for field in TRACE_FIELDS]
    time, voltage, current, capacity = traces
    if (time.size < 2 or any(trace.ndim != 1 or trace.shape != time.shape for trace in traces)
            or not all(np.isfinite(trace).all() for trace in traces)
            or np.any(np.diff(time) <= 0) or cycle.get("attribute") != "Cycling"):
        return result

    charge_starts = np.flatnonzero(current > 1)
    if charge_starts.size:
        start = int(charge_starts[0])
        rate, measured_current = actual_charge_rate(time, current, start)
        result.update(observed_charge_rate_c=rate, cc_current_a=measured_current)
        cutoff = np.flatnonzero((current[start:] > 0) & (current[start:] <= 0.2)
                                & (voltage[start:] >= 4.15))
        if cutoff.size and not np.any(current[start:start + cutoff[0]] < -0.1):
            result["charge_to_cutoff_s"] = float(time[start + cutoff[0]] - time[start])

    recorded = float(capacity.max())
    result["recorded_discharge_ah"] = recorded
    active = np.flatnonzero(current < -0.1)
    if active.size < 2:
        result["energy_status"] = "missing_discharge"
        return result
    median_current = float(np.median(current[active]))
    result["discharge_current_a"] = median_current
    if (abs(median_current + NOMINAL_AH) > 0.1 or abs(voltage[active[-1]] - 2.5) > 0.1
            or not np.all(np.diff(active) == 1)):
        result["energy_status"] = "incomplete_or_off_protocol_discharge"
        return result
    integrated, energy = discharge_integrals(time, voltage, current)
    result["integrated_discharge_ah"] = integrated
    if recorded <= 0 or energy <= 0 or abs(integrated - recorded) > max(0.01, 0.01 * recorded):
        result["energy_status"] = "capacity_mismatch"
    else:
        result.update(delivered_wh=energy, energy_status="ok")
    return result


def sustained_switches(audit):
    events = []
    for cell, group in audit.groupby("cell_id", sort=False):
        schedule = group.set_index("cycle").observed_charge_rate_c.to_dict()
        for cycle, rate in schedule.items():
            old = schedule.get(cycle - 1)
            if (old in RATES and rate in RATES and rate != old
                    and schedule.get(cycle - 2) == old and schedule.get(cycle + 1) == rate):
                events.append((cell, cycle, old, rate))
    return pd.DataFrame(events, columns=SWITCH_COLUMNS)


def audit_batteries():
    if not dataset.is_dir() and not dataset.is_file():
        raise FileNotFoundError(f"Expected extracted XJTU data or a ZIP archive at {dataset}.")
    order = [*PILOT_CELLS, *(cell for cell in cells.index if cell not in PILOT_CELLS)]
    results = []
    archive = ZipFile(dataset) if dataset.is_file() else nullcontext()
    with archive as bundle:
        paths = defaultdict(list)
        if bundle is not None:
            for file in bundle.namelist():
                if Path(file).suffix == ".pkl":
                    paths[Path(file).stem].append(file)
        for cell in order:
            if bundle is not None:
                if len(paths[cell]) != 1:
                    raise ValueError(f"Expected a single {cell}.pkl in {dataset}.")
                stream = bundle.open(paths[cell][0])
            else:
                stream = (dataset / f"{cell}.pkl").open("rb")
            with stream:
                data = XJTUUnpickler(stream).load()
            rate = int(cells.loc[cell, "charge_rate_c"])
            discharge_protocol = [step["rate_in_C"] for step in data["discharge_protocol"]]
            charge_protocol = [step["rate_in_C"] for step in data["charge_protocol"]]
            if (data["cell_id"] != cell or data["nominal_capacity_in_Ah"] != NOMINAL_AH
                    or discharge_protocol != [1.0] or charge_protocol != [rate]):
                raise ValueError(f"Unexpected fixed-protocol metadata for {cell}.")
            records = data["cycle_data"]
            cycles = [row["cycle_number"] for row in records]
            if cycles != list(range(1, len(cycles) + 1)) or len(cycles) < cells.loc[cell, "endpoint"]:
                raise ValueError(f"Incomplete or unordered raw cycle data for {cell}.")
            audited = [{"cell_id": cell, "cycle": cycle["cycle_number"], "charge_rate_c": rate,
                        **audit_cycle(cycle)} for cycle in records]
            results.extend(audited)
            if cell in PILOT_CELLS:
                pilot = {row["cycle"]: row for row in audited if row["cycle"] in ANCHORS}
                for anchor, row in pilot.items():
                    assert row["observed_charge_rate_c"] == rate and row["energy_status"] == "ok"
                    print(f"Pilot {cell}, cycle {anchor}: {row['cc_current_a']:.3f} A, "
                          f"{row['delivered_wh']:.3f} Wh")
                if set(pilot) != set(ANCHORS):
                    raise ValueError(f"Missing pilot anchors for {cell}.")
            del records, data
    audit = pd.DataFrame(results).sort_values(["cell_id", "cycle"]).reset_index(drop=True)
    unique_keys(audit, ["cell_id", "cycle"], "raw XJTU cycle audit")
    events = sustained_switches(audit)
    print(f"Audited {len(audit):,} raw cycles. Found {len(events)} sustained same-cell rate switches.")
    return audit, events


def check_audit_helpers():
    time = np.arange(5, dtype=float) * 360
    volts = np.full(5, 3.5)
    current = np.array([0, -2, -2, -2, 0], dtype=float)
    np.testing.assert_allclose(discharge_integrals(time, volts, current), (0.4, 1.4))
    assert discharge_integrals(time, volts, np.zeros(5)) is None
    demo = pd.DataFrame({"cell_id": ["demo"] * 6, "cycle": [1, 2, 3, 4, 5, 6],
                         "observed_charge_rate_c": [2, 2, 3, 3, 2, 3]})
    assert sustained_switches(demo).cycle.tolist() == [3]


check_audit_helpers()
with threadpool_limits(limits=1):
    cycle_audit, switch_audit = audit_batteries()

Pilot XJTU_2C_battery-7, cycle 50: 4.000 A, 7.081 Wh
Pilot XJTU_2C_battery-7, cycle 100: 4.000 A, 6.994 Wh
Pilot XJTU_3C_battery-11, cycle 50: 6.000 A, 6.832 Wh
Pilot XJTU_3C_battery-11, cycle 100: 6.000 A, 6.490 Wh
Audited 6,938 raw cycles. Found 0 sustained same-cell rate switches.


## Fit the fixed-protocol transition inside each fold

The state starts with measured SOH through the end of cycle 50 or 100. A Ridge model learns average per-cycle decline over observed ten-cycle training windows. Its inputs describe current SOH, ten-cycle decline and charge rate. The rollout updates the history with its own predictions. SOH remains at the 80% threshold after the first crossing.

The measured Notebook 07 fade curves and persistence provide baselines. The same training-fold voltage model converts predicted discharge capacity into Wh for all three models. Energy accumulates through the crossing cycle and stops afterward. Each 2-point SOH bin is supported when at least two training batteries contribute at that rate.

In [4]:
def band(soh):
    return 2.0 * np.floor(np.asarray(soh) / 2.0)


def transition_features(soh, trend, rate):
    base = np.column_stack((soh, np.maximum(95 - soh, 0), np.maximum(90 - soh, 0),
                            np.maximum(85 - soh, 0), trend))
    fast = (rate == 3).astype(float)
    return np.column_stack((base, fast, base * fast[:, None]))


def model_inputs():
    targets, energy, support = [], [], []
    for cell, info in cells.iterrows():
        end, rate = int(info.endpoint), int(info.charge_rate_c)
        raw = cycle_audit.loc[cycle_audit.cell_id.eq(cell)].set_index("cycle").reindex(range(1, end + 1))
        soh = measured_soh[cell].reindex(raw.index).to_numpy()
        verified = raw.observed_charge_rate_c.eq(rate).to_numpy()
        t = np.arange(min(ANCHORS), end - WINDOW + 1)
        start, future, past = soh[t - 1], soh[t + WINDOW - 1], soh[t - WINDOW - 1]
        valid = np.isfinite(start) & np.isfinite(future) & np.isfinite(past) & (start > EOL_SOH)
        verified_next_window = np.convolve(verified.astype(int), np.ones(WINDOW, dtype=int), "valid")
        valid &= verified_next_window[t] == WINDOW
        targets.append(pd.DataFrame({
            "cell_id": cell, "charge_rate_c": rate, "soh": start[valid],
            "trend": np.maximum((past[valid] - start[valid]) / WINDOW, 0),
            "fade": np.maximum((start[valid] - future[valid]) / WINDOW, 0),
        }))

        steps = np.arange(min(ANCHORS), end)
        eligible = verified[steps] & np.isfinite(soh[steps - 1]) & (soh[steps - 1] > EOL_SOH)
        support.append(pd.DataFrame({
            "cell_id": cell, "charge_rate_c": rate, "soh_bin": band(soh[steps[eligible] - 1]),
        }))
        good = eligible & np.isfinite(soh[steps]) & (raw.energy_status.to_numpy()[steps] == "ok")
        eligible_positions = steps[good]
        energy.append(pd.DataFrame({
            "cell_id": cell, "charge_rate_c": rate, "soh_bin": band(soh[eligible_positions]),
            "mean_voltage": (raw.delivered_wh / raw.integrated_discharge_ah).to_numpy()[eligible_positions],
        }))
    return tuple(pd.concat(rows, ignore_index=True) for rows in (targets, energy, support))


def prepare_starts(cell_ids):
    contexts, histories = [], []
    for cell in sorted(cell_ids):
        info, trace = cells.loc[cell], measured_soh[cell]
        for anchor in ANCHORS:
            hist = trace.reindex(range(anchor - WINDOW, anchor + 1)).to_numpy()
            if not np.isfinite(hist[-1]):
                continue
            if not np.isfinite(hist).all():
                # Only earlier measurements contribute to the trend at initialization.
                hist = pd.Series(hist).interpolate(limit_direction="both").to_numpy()
            contexts.append({"cell_id": cell, "fold": int(info.fold), "charge_rate_c": int(info.charge_rate_c),
                             "anchor_cycle": anchor, "endpoint": int(info.endpoint)})
            histories.append(hist)
    return pd.DataFrame(contexts), np.asarray(histories)


def fit_ridge(train, alpha):
    charge = train.charge_rate_c.to_numpy()
    x = transition_features(train.soh.to_numpy(), train.trend.to_numpy(), charge)
    weights = 1 / train.groupby("cell_id").cell_id.transform("size").to_numpy(dtype=float)
    weights = weights / weights.mean()
    center = np.average(x, weights=weights, axis=0)
    scale = np.sqrt(np.average((x - center) ** 2, weights=weights, axis=0))
    scale = np.where(scale > 1e-10, scale, 1.0)
    model = Ridge(alpha=alpha)
    model.fit((x - center) / scale, train.fade, sample_weight=weights)
    coef = model.coef_ / scale
    bias = float(model.intercept_ - center @ coef)
    upper = {int(rate): float(part.fade.max()) for rate, part in train.groupby("charge_rate_c")}
    return coef, bias, upper


def voltage_profiles(training_energy):
    med = training_energy.groupby(["charge_rate_c", "soh_bin", "cell_id"]).mean_voltage.median()
    med = med.groupby(["charge_rate_c", "soh_bin"]).median().reset_index()
    result = {}
    for rate in RATES:
        profile = med.loc[med.charge_rate_c.eq(rate)].sort_values("soh_bin")
        if profile.empty:
            raise ValueError(f"No usable voltage labels in the {rate}C training data.")
        result[rate] = (profile.soh_bin.to_numpy() + 1, profile.mean_voltage.to_numpy())
    return result


def interpolate_by_rate(soh, rates, profiles):
    out = np.empty_like(soh, dtype=float)
    for rate in RATES:
        choose = rates == rate
        out[choose] = np.interp(soh[choose], *profiles[rate])
    return out


def rollout(history, rates, model, *, ridge=None, profiles=None, volts=None, supported_bins=None, steps=MAX_STEPS):
    if not np.isin(rates, RATES).all():
        raise ValueError("The twin supports the recorded 2C and 3C charge rates.")
    values = np.empty((len(rates), WINDOW + steps + 1))
    values[:, :WINDOW + 1] = history
    eol = np.where(history[:, -1] <= EOL_SOH, 0, -1)
    energy = np.zeros((len(rates), steps + 1)) if volts is not None else None
    supported = np.ones((len(rates), steps + 1), dtype=bool) if supported_bins is not None else None

    if model == "ridge":
        coef, bias, max_fade = ridge
        upper = np.array([max_fade[rate] for rate in rates])
    for step in range(1, steps + 1):
        current = values[:, WINDOW + step - 1]
        active = eol < 0
        if model == "ridge":
            trend = np.maximum((values[:, step - 1] - current) / WINDOW, 0)
            fade = np.clip(transition_features(current, trend, rates) @ coef + bias, 0, upper)
        elif model == "nb07":
            fade = interpolate_by_rate(current, rates, profiles)
        elif model == "persistence":
            fade = np.zeros_like(current)
        else:
            raise ValueError(f"Unknown transition model {model}.")
        next_soh = np.where(active, np.maximum(EOL_SOH, current - fade), current)
        values[:, WINDOW + step] = next_soh
        eol[active & (next_soh <= EOL_SOH)] = step
        if volts is not None:
            voltage = interpolate_by_rate(next_soh, rates, volts)
            energy[:, step] = energy[:, step - 1] + np.where(active, NOMINAL_AH * next_soh / 100 * voltage, 0)
        if supported is not None:
            current_bins = band(current)
            inside = np.array([value in supported_bins[rate] for rate, value in zip(rates, current_bins)])
            supported[:, step] = supported[:, step - 1] & (~active | inside)
    return values[:, WINDOW:], eol, energy, supported


def select_alpha(training_cells, outer_fold):
    rows = []
    inner_folds = sorted(cells.loc[list(training_cells), "fold"].unique())
    for validation_fold in inner_folds:
        validation = cells.index[cells.fold.eq(validation_fold)].intersection(training_cells)
        training = training_cells.difference(validation)
        context, history = prepare_starts(validation)
        rates = context.charge_rate_c.to_numpy()
        table = transition_training.loc[transition_training.cell_id.isin(training)]
        for alpha in ALPHAS:
            path, _, _, _ = rollout(history, rates, "ridge", ridge=fit_ridge(table, alpha), steps=max(HORIZONS))
            for index, row in enumerate(context.itertuples(index=False)):
                for horizon in (10, 50):
                    target = row.anchor_cycle + horizon
                    actual = measured_soh[row.cell_id].get(target, np.nan)
                    if target <= row.endpoint and np.isfinite(actual):
                        rows.append((alpha, row.charge_rate_c, row.anchor_cycle, horizon,
                                     abs(path[index, horizon] - actual)))
    scores = pd.DataFrame(rows, columns=["alpha", "charge_rate_c", "anchor_cycle", "horizon", "abs_error_pp"])
    if set(scores.alpha) != set(ALPHAS):
        raise ValueError(f"No complete inner validation scores for fold {outer_fold}.")
    score = scores.groupby(["alpha", "charge_rate_c", "anchor_cycle", "horizon"]).abs_error_pp.mean()
    score = score.groupby("alpha").mean()
    alpha = min(ALPHAS, key=lambda value: (score[value], -value))
    return alpha, [{"outer_fold": outer_fold, "alpha": value, "inner_mae_pp": score[value],
                    "selected": value == alpha} for value in ALPHAS]


transition_training, energy_training, state_support = model_inputs()

In [5]:
MODEL_NAMES = {"persistence": "Persistence", "nb07": "Notebook 07", "ridge": "Ridge"}
KEYS = ["cell_id", "fold", "charge_rate_c", "anchor_cycle"]


def predict_factual():
    trajectories, eol_rows, selection, support_rows = [], [], [], []
    traces = {cell: table.set_index("cycle") for cell, table in cycle_audit.groupby("cell_id")}
    nb07_prior = nb07_starts.set_index(["cell_id", "landmark_cycle"])
    nb07_values = nb07_curves.set_index(["cell_id", "landmark_cycle", "cycle"]).sort_index()
    for fold in sorted(cells.fold.unique()):
        test = cells.index[cells.fold.eq(fold)]
        training = cells.index.difference(test)
        contexts, history = prepare_starts(test)
        rates = contexts.charge_rate_c.to_numpy()
        alpha, scores = select_alpha(training, fold)
        selection.extend(scores)
        ridge = fit_ridge(transition_training.loc[transition_training.cell_id.isin(training)], alpha)
        volts = voltage_profiles(energy_training.loc[energy_training.cell_id.isin(training)])
        profiles = {}
        for rate in RATES:
            fit = nb07_fade.loc[nb07_fade.fold.eq(fold) & nb07_fade.charge_rate_c.eq(rate)]
            fit = fit.sort_values("bin_bottom_soh_pct")
            profiles[rate] = (fit.bin_bottom_soh_pct.to_numpy() + 1, fit.model_fade_pp_per_cycle.to_numpy())

        support = state_support.loc[state_support.cell_id.isin(training)]
        counts = support.groupby(["soh_bin", "charge_rate_c"]).cell_id.nunique().unstack(fill_value=0)
        counts = counts.reindex(columns=RATES, fill_value=0)
        supported = {rate: frozenset(counts.index[counts[rate].ge(MIN_CELLS_IN_BIN)]) for rate in RATES}
        for bin_soh, row in counts.iterrows():
            support_rows.append({
                "fold": int(fold), "soh_bin_start_pct": bin_soh, "soh_bin_end_pct": bin_soh + 2,
                "training_cells_2c": int(row[2]), "training_cells_3c": int(row[3]),
                "both_rates_supported": bin_soh in supported[2] and bin_soh in supported[3],
            })

        for model in MODEL_NAMES:
            path, crossing, cumulative, valid_bins = rollout(
                history, rates, model, ridge=ridge, profiles=profiles, volts=volts, supported_bins=supported)
            for index, row in enumerate(contexts.itertuples(index=False)):
                cell, anchor, end = row.cell_id, row.anchor_cycle, row.endpoint
                observed_eol = cells.loc[cell, "observed_eol_cycle"]
                predicted_eol = anchor + crossing[index] if crossing[index] >= 0 else None
                key = {name: getattr(row, name) for name in KEYS}
                target = np.arange(anchor + 1, end + 1)
                horizon = target - anchor
                if len(horizon) and horizon[-1] > MAX_STEPS:
                    target = target[horizon <= MAX_STEPS]
                    horizon = target - anchor
                raw = traces[cell].reindex(target)
                factual_rate = raw.observed_charge_rate_c
                compatible = np.logical_and.accumulate(
                    (factual_rate.isna() | factual_rate.eq(row.charge_rate_c)).to_numpy())
                verified_rate = np.logical_and.accumulate(factual_rate.eq(row.charge_rate_c).to_numpy())
                actual_soh = measured_soh[cell].reindex(target).to_numpy()
                actual_wh = raw.delivered_wh.to_numpy(dtype=float)
                if model == "nb07":
                    saved = nb07_values.loc[(cell, anchor)].predicted_soh_pct.reindex(target).to_numpy()
                    np.testing.assert_allclose(path[index, horizon], saved, rtol=0, atol=1e-8)
                    prior = nb07_prior.loc[(cell, anchor)]
                    np.testing.assert_allclose(history[index, -1], prior.initial_soh_pct, rtol=0, atol=1e-8)
                    prior_eol = prior.predicted_rul_cycles
                    if (pd.notna(prior_eol) and predicted_eol != anchor + int(prior_eol)
                            or pd.isna(prior_eol) and predicted_eol is not None):
                        raise ValueError(f"Notebook 07 baseline crossing differs for {cell} at {anchor}.")

                eol_rows.append({
                    **key, "model": model, "observed_eol_cycle": observed_eol,
                    "last_observed_cycle": int(cells.loc[cell, "last_observed_cycle"]),
                    "event_status": cells.loc[cell, "event_status"], "predicted_eol_cycle": predicted_eol,
                    "forecast_status": "crossed" if predicted_eol is not None else "no_crossing",
                    "recorded_rate_compatible": bool(compatible.all()),
                    "supported_to_predicted_eol": bool(valid_bins[index, crossing[index]]) if predicted_eol is not None else None,
                })
                trajectories.append(pd.DataFrame({
                    **key, "model": model, "target_cycle": target, "horizon_cycles": horizon,
                    "observed_soh_pct": actual_soh, "predicted_soh_pct": path[index, horizon],
                    "observed_delivered_wh": actual_wh,
                    "predicted_delivered_wh": np.diff(cumulative[index], prepend=0)[horizon],
                    "observed_cumulative_wh": np.cumsum(actual_wh),
                    "predicted_cumulative_wh": cumulative[index, horizon],
                    "recorded_rate_compatible": compatible, "recorded_rate_verified": verified_rate,
                    "training_support_through_target": valid_bins[index, horizon],
                }))
    predictions = pd.concat(trajectories, ignore_index=True)
    key = ["cell_id", "anchor_cycle", "model"]
    life = pd.DataFrame(eol_rows)
    unique_keys(predictions, [*key, "target_cycle"], "factual forecasts")
    unique_keys(life, key, "forecast EOLs")
    return predictions, life, pd.DataFrame(selection), pd.DataFrame(support_rows)


def score_group(rows, actual_field, predicted_field, target_field, keys):
    eligible = rows.loc[rows.recorded_rate_compatible & rows[actual_field].notna()]
    scored = eligible.loc[eligible[predicted_field].notna()].copy()
    result = {
        **keys, "eligible_points": len(eligible), "scored_points": len(scored),
        "scored_cells": scored.cell_id.nunique(),
        "mae": None, "signed_bias": None, "worst": None, "worst_cell": None,
        "worst_anchor_cycle": None, "worst_target_cycle": None,
        "support_pct": None,
    }
    if scored.empty:
        return result
    signed = scored[predicted_field] - scored[actual_field]
    scored["absolute"] = signed.abs()
    scored["signed"] = signed
    worst = scored.loc[scored.absolute.idxmax()]
    result.update(mae=scored.groupby("cell_id").absolute.mean().mean(),
                  signed_bias=scored.groupby("cell_id").signed.mean().mean(),
                  worst=worst.absolute, worst_cell=worst.cell_id,
                  worst_anchor_cycle=int(worst.anchor_cycle), worst_target_cycle=int(worst[target_field]))
    support_column = "training_support_through_target" if "training_support_through_target" in scored else "supported_to_predicted_eol"
    if support_column in scored:
        result["support_pct"] = 100 * scored[support_column].astype(float).mean()
    return result


def score_forecasts(predictions, life):
    rows = []
    group_keys = ["charge_rate_c", "anchor_cycle", "model"]
    endpoint = predictions.cell_id.map(cells.observed_eol_cycle)
    late = predictions.loc[endpoint.notna() & predictions.target_cycle.between(endpoint - 19, endpoint)]
    for (rate, anchor, model), trace in predictions.groupby(group_keys):
        context = dict(charge_rate_c=rate, anchor_cycle=anchor, model=model)
        for horizon in HORIZONS:
            paired = trace.loc[trace.horizon_cycles.eq(horizon)]
            rows.append(score_group(paired, "observed_soh_pct", "predicted_soh_pct", "target_cycle",
                                    {**context, "metric": "soh_pp", "window": str(horizon)}))
            rows.append(score_group(paired, "observed_cumulative_wh", "predicted_cumulative_wh", "target_cycle",
                                    {**context, "metric": "cumulative_wh", "window": str(horizon)}))
        closing = late.loc[late.model.eq(model) & late.charge_rate_c.eq(rate) & late.anchor_cycle.eq(anchor)]
        rows.append(score_group(closing, "observed_soh_pct", "predicted_soh_pct", "target_cycle",
                                {**context, "metric": "soh_pp", "window": "last20"}))
    for (rate, anchor, model), group in life.groupby(group_keys):
        context = dict(charge_rate_c=rate, anchor_cycle=anchor, model=model, metric="eol_cycles", window="eol")
        rows.append(score_group(group, "observed_eol_cycle", "predicted_eol_cycle", "observed_eol_cycle", context))
    return pd.DataFrame(rows)


def rolling_benchmark(predictions):
    try:
        with saved_csv(NB11, "soh_predictions.csv") as rows:
            rolling = rows.loc[rows.origin_cycle.isin(ANCHORS) & rows.horizon_cycles.isin(HORIZONS)].copy()
    except FileNotFoundError:
        return pd.DataFrame()
    rolling = rolling.rename(columns={"origin_cycle": "anchor_cycle", "selected_soh_pct": "rolling_soh_pct"})
    keys = [*KEYS, "target_cycle", "horizon_cycles"]
    ridge = predictions.loc[predictions.model.eq("ridge") & predictions.horizon_cycles.isin(HORIZONS),
                            [*keys, "observed_soh_pct", "predicted_soh_pct", "recorded_rate_compatible"]]
    paired = ridge.merge(rolling[[*keys, "observed_soh_pct", "rolling_soh_pct"]],
                         on=keys, suffixes=("", "_rolling"), validate="one_to_one")
    np.testing.assert_allclose(paired.observed_soh_pct, paired.observed_soh_pct_rolling, rtol=0, atol=1e-9)
    result = []
    for (rate, anchor, horizon), group in paired.groupby(["charge_rate_c", "anchor_cycle", "horizon_cycles"]):
        for name, field in (("ridge", "predicted_soh_pct"), ("rolling_nb11", "rolling_soh_pct")):
            result.append(score_group(group, "observed_soh_pct", field, "target_cycle",
                                      dict(charge_rate_c=rate, anchor_cycle=anchor, horizon_cycles=horizon, model=name)))
    return pd.DataFrame(result)


with threadpool_limits(limits=1):
    trajectories, eol_predictions, model_selection, protocol_support = predict_factual()
metrics = score_forecasts(trajectories, eol_predictions)
rolling_reference = rolling_benchmark(trajectories)

In [6]:
def metric_text(row, precision=3):
    if not row.scored_points:
        return f"— ({row.scored_points}/{row.eligible_points})"
    return (f"{row.mae:.{precision}f} / {row.worst:.{precision}f} "
            f"({row.scored_points}/{row.eligible_points})")


def report():
    result = [
        "# XJTU action-conditioned twin validation", "",
        "This report evaluates constant-rate forecasts on held-out batteries. Each start uses measured SOH "
        "and ten cycles of past SOH through the completed anchor cycle. Subsequent rollout steps use predicted SOH. "
        "The rows summarize errors on the recorded protocol. MAE gives each battery equal weight.", "",
        f"The raw audit found **{len(switch_audit)} sustained same-battery 2C↔3C switches** "
        "in the 23-battery cohort. A switch needs two adjacent classified cycles on each side. "
        "The action overlap table describes where both rates have training observations. "
        "Actual switching effects require independent evidence.", "",
        "## Raw data coverage", "",
        "| Rate | Batteries | Verified EOL | Rate classified as expected | Valid discharge energy | Median charge-to-cutoff, min |",
        "| --- | ---: | ---: | ---: | ---: | ---: |",
    ]
    for rate in RATES:
        raw = cycle_audit.loc[cycle_audit.charge_rate_c.eq(rate)]
        cohort = cells.loc[cells.charge_rate_c.eq(rate)]
        agreeing = raw.observed_charge_rate_c.eq(rate)
        mins = raw.loc[agreeing].groupby("cell_id").charge_to_cutoff_s.median().median() / 60
        result.append(f"| {rate}C | {len(cohort)} | {cohort.observed_eol_cycle.notna().sum()} | "
                      f"{agreeing.sum()}/{len(raw)} | {raw.energy_status.eq('ok').sum()}/{len(raw)} | {mins:.1f} |")
    result += ["", "## Held-out forecast errors", "",
               "Each entry is **MAE / worst absolute error (scored/eligible)**. SOH uses percentage points. "
               "Cumulative energy uses Wh from the cycle after the anchor through the target. Its forecast "
               "stops accumulating after predicted EOL. EOL error is in cycles. SOH holds at 80% after predicted EOL. "
               "A missing energy measurement invalidates cumulative energy targets that include it.", ""]
    indexed = metrics.set_index(["charge_rate_c", "anchor_cycle", "metric", "window", "model"])
    for metric, title, windows in (
        ("soh_pp", "SOH", ("1", "10", "50", "last20")),
        ("cumulative_wh", "Cumulative energy (Wh)", ("1", "10", "50")),
        ("eol_cycles", "EOL (cycles)", ("eol",)),
    ):
        result += [f"### {title}", "",
                   "| Rate | Anchor | Window | Persistence | Notebook 07 | Ridge |",
                   "| --- | ---: | --- | ---: | ---: | ---: |"]
        for rate in RATES:
            for anchor in ANCHORS:
                for window in windows:
                    rows = [indexed.loc[(rate, anchor, metric, window, model)] for model in MODEL_NAMES]
                    values = " | ".join(metric_text(row, 2 if metric == "eol_cycles" else 3) for row in rows)
                    result.append(f"| {rate}C | {anchor} | {window} | {values} |")
        result.append("")

    result += ["### Largest Ridge errors at cycle 50 or 100 starts", "",
               "| Metric | Worst error | Cell | Anchor | Target or observed EOL cycle |",
               "| --- | ---: | --- | ---: | ---: |"]
    for metric in ("soh_pp", "cumulative_wh", "eol_cycles"):
        scope = metrics.loc[metrics.model.eq("ridge") & metrics.metric.eq(metric) & metrics.worst.notna()]
        if scope.empty:
            continue
        worst = scope.loc[scope.worst.idxmax()]
        result.append(f"| {metric} | {worst.worst:.3f} | {worst.worst_cell} | "
                      f"{int(worst.worst_anchor_cycle)} | {int(worst.worst_target_cycle)} |")

    result += ["", "## Validation outcome", ""]
    compare = metrics.loc[metrics.metric.eq("soh_pp") & metrics.window.isin(["10", "50"])].pivot(
        index=["charge_rate_c", "anchor_cycle", "window"], columns="model", values="mae")
    better = compare.ridge.lt(compare.nb07)
    result.append(f"- Ridge improved SOH MAE over Notebook 07 in **{better.sum()}/{len(better)}** "
                  "paired 10- and 50-cycle groups.")
    if better.any():
        cases = ", ".join(f"{rate}C at {anchor} +{horizon}" for rate, anchor, horizon in compare.index[better])
        result.append(f"- The improved groups were {cases}.")
    common = protocol_support.loc[protocol_support.both_rates_supported].groupby("fold").soh_bin_start_pct.apply(list)
    shared_bins = set.intersection(*(set(common.get(fold, [])) for fold in cells.fold.unique()))
    spans = ", ".join(f"{level:g}–{level + 2:g}%" for level in sorted(shared_bins))
    result.append(f"- Folds share training support for both rates in these two-point SOH bins: {spans or 'none'}.")
    result += [
        "- EOL errors use verified events and report prediction coverage. Right-censored batteries contribute "
        "only their available SOH and energy labels.",
        "- The validation applies to fixed rates on the observed protocols. The switch audit records candidate "
        "events separately from fixed-rate scoring.",
    ]
    if not rolling_reference.empty:
        result.append("- rolling_reference.csv gives the Notebook 11 comparison on aligned targets with its own "
                      "information at the origin. The rolling estimates did not enter fitting or selection.")
    return "\n".join(result) + "\n"


summary_text = report()
output.mkdir(parents=True, exist_ok=True)
tables = {
    "cycle_audit.csv": cycle_audit,
    "observed_switches.csv": switch_audit,
    "action_support.csv": protocol_support,
    "predictions.csv": trajectories,
    "eol_predictions.csv": eol_predictions,
    "metrics.csv": metrics,
    "model_selection.csv": model_selection,
}
if not rolling_reference.empty:
    tables["rolling_reference.csv"] = rolling_reference

for filename, frame in tables.items():
    frame.to_csv(output / filename, index=False)
(output / "summary.md").write_text(summary_text, encoding="utf-8")
settings = {
    "dataset": "BatteryLife v12 XJTU, fixed 2C and 3C with 1C discharge",
    "pilot_cells": PILOT_CELLS, "anchor_cycles": ANCHORS, "horizons": HORIZONS,
    "charge_rates_c": RATES, "eol_soh_pct": EOL_SOH, "nominal_ah": NOMINAL_AH,
    "max_rollout_steps": MAX_STEPS, "minimum_cells_per_rate_bin": MIN_CELLS_IN_BIN,
    "ridge_alphas": ALPHAS, "training_target_window_cycles": WINDOW,
    "algorithm_inputs": [NB01, NB07], "rolling_reference": not rolling_reference.empty,
}
(output / "run_config.json").write_text(json.dumps(settings, indent=2) + "\n", encoding="utf-8")
bundle_path = output.with_suffix(".zip")
with ZipFile(bundle_path, "w", compression=ZIP_DEFLATED) as bundle:
    for name in (*tables, "run_config.json", "summary.md"):
        bundle.write(output / name, name)

from IPython import get_ipython
from IPython.display import Markdown, display

if get_ipython() is not None:
    display(Markdown(summary_text))
print(f"Saved {bundle_path}")

# XJTU action-conditioned twin validation

This report evaluates constant-rate forecasts on held-out batteries. Each start uses measured SOH and ten cycles of past SOH through the completed anchor cycle. Subsequent rollout steps use predicted SOH. The rows summarize errors on the recorded protocol. MAE gives each battery equal weight.

The raw audit found **0 sustained same-battery 2C↔3C switches** in the 23-battery cohort. A switch needs two adjacent classified cycles on each side. The action overlap table describes where both rates have training observations. Actual switching effects require independent evidence.

## Raw data coverage

| Rate | Batteries | Verified EOL | Rate classified as expected | Valid discharge energy | Median charge-to-cutoff, min |
| --- | ---: | ---: | ---: | ---: | ---: |
| 2C | 8 | 7 | 3211/3211 | 3210/3211 | 42.5 |
| 3C | 15 | 15 | 3727/3727 | 3727/3727 | 33.8 |

## Held-out forecast errors

Each entry is **MAE / worst absolute error (scored/eligible)**. SOH uses percentage points. Cumulative energy uses Wh from the cycle after the anchor through the target. Its forecast stops accumulating after predicted EOL. EOL error is in cycles. SOH holds at 80% after predicted EOL. A missing energy measurement invalidates cumulative energy targets that include it.

### SOH

| Rate | Anchor | Window | Persistence | Notebook 07 | Ridge |
| --- | ---: | --- | ---: | ---: | ---: |
| 2C | 50 | 1 | 0.025 / 0.050 (8/8) | 0.025 / 0.044 (8/8) | 0.024 / 0.040 (8/8) |
| 2C | 50 | 10 | 0.150 / 0.200 (8/8) | 0.245 / 0.352 (8/8) | 0.286 / 0.355 (8/8) |
| 2C | 50 | 50 | 0.444 / 0.900 (8/8) | 0.373 / 0.765 (8/8) | 0.818 / 1.134 (8/8) |
| 2C | 50 | last20 | 16.766 / 18.900 (140/140) | 1.508 / 3.400 (140/140) | 1.305 / 3.400 (140/140) |
| 2C | 100 | 1 | 0.062 / 0.100 (8/8) | 0.048 / 0.085 (8/8) | 0.038 / 0.079 (8/8) |
| 2C | 100 | 10 | 0.031 / 0.050 (8/8) | 0.134 / 0.210 (8/8) | 0.268 / 0.353 (8/8) |
| 2C | 100 | 50 | 0.544 / 0.800 (8/8) | 0.423 / 0.869 (8/8) | 1.018 / 1.288 (8/8) |
| 2C | 100 | last20 | 16.309 / 18.500 (140/140) | 1.120 / 3.400 (140/140) | 1.221 / 3.400 (140/140) |
| 3C | 50 | 1 | 0.060 / 0.200 (15/15) | 0.076 / 0.219 (15/15) | 0.061 / 0.202 (15/15) |
| 3C | 50 | 10 | 0.160 / 0.500 (15/15) | 0.158 / 0.392 (15/15) | 0.102 / 0.300 (15/15) |
| 3C | 50 | 50 | 1.637 / 4.750 (15/15) | 0.648 / 2.343 (15/15) | 0.781 / 1.645 (15/15) |
| 3C | 50 | last20 | 14.337 / 19.450 (300/300) | 3.614 / 12.903 (300/300) | 7.818 / 19.450 (300/300) |
| 3C | 100 | 1 | 0.063 / 0.200 (15/15) | 0.040 / 0.137 (15/15) | 0.049 / 0.137 (15/15) |
| 3C | 100 | 10 | 0.707 / 2.300 (15/15) | 0.220 / 0.810 (15/15) | 0.280 / 0.474 (15/15) |
| 3C | 100 | 50 | 2.615 / 8.350 (13/13) | 0.454 / 2.791 (13/13) | 4.525 / 7.287 (13/13) |
| 3C | 100 | last20 | 12.700 / 18.250 (300/300) | 2.558 / 7.995 (300/300) | 2.150 / 6.700 (300/300) |

### Cumulative energy (Wh)

| Rate | Anchor | Window | Persistence | Notebook 07 | Ridge |
| --- | ---: | --- | ---: | ---: | ---: |
| 2C | 50 | 1 | 0.016 / 0.022 (8/8) | 0.016 / 0.023 (8/8) | 0.016 / 0.023 (8/8) |
| 2C | 50 | 10 | 0.184 / 0.255 (8/8) | 0.226 / 0.313 (8/8) | 0.235 / 0.317 (8/8) |
| 2C | 50 | 50 | 0.581 / 1.580 (8/8) | 1.587 / 2.167 (8/8) | 2.414 / 2.983 (8/8) |
| 2C | 100 | 1 | 0.010 / 0.018 (8/8) | 0.009 / 0.017 (8/8) | 0.008 / 0.016 (8/8) |
| 2C | 100 | 10 | 0.037 / 0.088 (8/8) | 0.052 / 0.120 (8/8) | 0.098 / 0.169 (8/8) |
| 2C | 100 | 50 | 0.856 / 1.433 (8/8) | 0.974 / 1.618 (8/8) | 2.392 / 2.859 (8/8) |
| 3C | 50 | 1 | 0.011 / 0.032 (15/15) | 0.012 / 0.033 (15/15) | 0.011 / 0.032 (15/15) |
| 3C | 50 | 10 | 0.101 / 0.207 (15/15) | 0.108 / 0.234 (15/15) | 0.093 / 0.200 (15/15) |
| 3C | 50 | 50 | 2.366 / 6.852 (15/15) | 1.228 / 2.664 (15/15) | 1.273 / 3.187 (15/15) |
| 3C | 100 | 1 | 0.006 / 0.023 (15/15) | 0.008 / 0.034 (15/15) | 0.010 / 0.039 (15/15) |
| 3C | 100 | 10 | 0.236 / 0.548 (15/15) | 0.064 / 0.193 (15/15) | 0.156 / 0.353 (15/15) |
| 3C | 100 | 50 | 4.611 / 12.559 (13/13) | 0.759 / 3.253 (13/13) | 6.080 / 9.378 (13/13) |

### EOL (cycles)

| Rate | Anchor | Window | Persistence | Notebook 07 | Ridge |
| --- | ---: | --- | ---: | ---: | ---: |
| 2C | 50 | eol | — (0/7) | 42.43 / 95.00 (7/7) | 53.86 / 91.00 (7/7) |
| 2C | 100 | eol | — (0/7) | 28.86 / 58.00 (7/7) | 27.00 / 53.00 (7/7) |
| 3C | 50 | eol | — (0/15) | 35.60 / 93.00 (15/15) | 43.56 / 148.00 (9/15) |
| 3C | 100 | eol | — (0/15) | 23.53 / 70.00 (15/15) | 79.47 / 154.00 (15/15) |

### Largest Ridge errors at cycle 50 or 100 starts

| Metric | Worst error | Cell | Anchor | Target or observed EOL cycle |
| --- | ---: | --- | ---: | ---: |
| soh_pp | 19.450 | XJTU_3C_battery-2 | 50 | 291 |
| cumulative_wh | 9.378 | XJTU_3C_battery-4 | 100 | 150 |
| eol_cycles | 154.000 | XJTU_3C_battery-4 | 100 | 321 |

## Validation outcome

- Ridge improved SOH MAE over Notebook 07 in **1/8** paired 10- and 50-cycle groups.
- The improved groups were 3C at 50 +10.
- Folds share training support for both rates in these two-point SOH bins: 80–82%, 82–84%, 84–86%, 86–88%, 88–90%, 90–92%, 92–94%, 94–96%, 96–98%, 98–100%.
- EOL errors use verified events and report prediction coverage. Right-censored batteries contribute only their available SOH and energy labels.
- The validation applies to fixed rates on the observed protocols. The switch audit records candidate events separately from fixed-rate scoring.
- rolling_reference.csv gives the Notebook 11 comparison on aligned targets with its own information at the origin. The rolling estimates did not enter fitting or selection.


Saved /Users/sakibsmacbookm5/Documents/Python work/research-project/SICIP Final Project/reports/lifeguard_action_conditioned_twin_results.zip
